<a href="https://colab.research.google.com/github/AlanSojan15/ML_Lab/blob/Lab8/2547208_Lab8_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import OneHotEncoder

# =====================================================================
# TASK 1: DATA PREPROCESSING
# =====================================================================

# Standard Play Tennis Dataset
df = pd.read_csv('https://docs.google.com/spreadsheets/d/1mnoUgK8YxInzoDQ7P7iBM1HeZ8tcnUSvMU_H1OWndFA/export?format=csv&gid=0')

# Separate input features and target variable
X = df[['Outlook', 'Temperature', 'Humidity', 'Wind']]
y = df['Play Tennis']

# Encode Features using OrdinalEncoder (required for CategoricalNB)
feature_encoder = OrdinalEncoder()
X_encoded = feature_encoder.fit_transform(X)

# Encode Target Variable using LabelEncoder
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# Save feature categories for single-sample query transformation
categories = feature_encoder.categories_

# =====================================================================
# TASK 2: DATASET PARTITIONING
# =====================================================================
# Stratify ensures both classes are represented in train and test sets given small sample size
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

# =====================================================================
# TASK 3: NAIVE BAYES MODEL TRAINING & EVALUATION
# =====================================================================
gnb = CategoricalNB()
gnb.fit(X_train, y_train)

y_pred_nb = gnb.predict(X_test)
accuracy_nb = accuracy_score(y_test, y_pred_nb)

print("="*60)
print("TASK 3: CATEGORICAL NAIVE BAYES EVALUATION")
print("="*60)
print(f"Overall Model Accuracy: {accuracy_nb * 100:.2f}%\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, target_names=target_encoder.classes_))

# =====================================================================
# TASK 4: SINGLE-SAMPLE INFERENCE
# =====================================================================
# Sample query: Outlook: Sunny, Temp: Cool, Humidity: High, Wind: Strong
sample_query = pd.DataFrame([['Sunny', 'Cool', 'High', 'Strong']],
                            columns=['Outlook', 'Temperature', 'Humidity', 'Wind'])

# Transform query using fitted feature encoder
sample_encoded = feature_encoder.transform(sample_query)

# Prediction and Probabilities
sample_pred_nb = gnb.predict(sample_encoded)
sample_proba_nb = gnb.predict_proba(sample_encoded)

pred_label_nb = target_encoder.inverse_transform(sample_pred_nb)[0]

print("="*60)
print("TASK 4: SINGLE-SAMPLE INFERENCE (CATEGORICAL NAIVE BAYES)")
print("="*60)
print(f"Input Query: Outlook=Sunny, Temperature=Cool, Humidity=High, Wind=Strong")
print(f"Predicted Class: {pred_label_nb}")
print(f"Class Probabilities -> No: {sample_proba_nb[0][0]:.4f}, Yes: {sample_proba_nb[0][1]:.4f}\n")

# =====================================================================
# TASK 5: MODEL COMPARISON
# =====================================================================
# For Logistic Regression and SVM, One-Hot Encoding performs better on nominal categorical data
ohe = OneHotEncoder(sparse_output=False)
X_ohe = ohe.fit_transform(X)
X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(
    X_ohe, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)
sample_ohe = ohe.transform(sample_query)

# 1. Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt_model.predict(X_test))
dt_pred = target_encoder.inverse_transform(dt_model.predict(sample_encoded))[0]
dt_proba = dt_model.predict_proba(sample_encoded)[0]

# 2. Logistic Regression Classifier
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_ohe, y_train_ohe)
lr_acc = accuracy_score(y_test_ohe, lr_model.predict(X_test_ohe))
lr_pred = target_encoder.inverse_transform(lr_model.predict(sample_ohe))[0]
lr_proba = lr_model.predict_proba(sample_ohe)[0]

# 3. Support Vector Machine (SVM with probability estimation enabled)
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train_ohe, y_train_ohe)
svm_acc = accuracy_score(y_test_ohe, svm_model.predict(X_test_ohe))
svm_pred = target_encoder.inverse_transform(svm_model.predict(sample_ohe))[0]
svm_proba = svm_model.predict_proba(sample_ohe)[0]

# Build Comparison Table
comparison_df = pd.DataFrame({
    'Model': ['Categorical Naive Bayes', 'Decision Tree', 'Logistic Regression', 'Support Vector Machine'],
    'Test Accuracy': [f"{accuracy_nb*100:.1f}%", f"{dt_acc*100:.1f}%", f"{lr_acc*100:.1f}%", f"{svm_acc*100:.1f}%"],
    'Sample Prediction': [pred_label_nb, dt_pred, lr_pred, svm_pred],
    'P(No)': [f"{sample_proba_nb[0][0]:.4f}", f"{dt_proba[0]:.4f}", f"{lr_proba[0]:.4f}", f"{svm_proba[0]:.4f}"],
    'P(Yes)': [f"{sample_proba_nb[0][1]:.4f}", f"{dt_proba[1]:.4f}", f"{lr_proba[1]:.4f}", f"{svm_proba[1]:.4f}"]
})

print("="*60)
print("TASK 5: MODEL COMPARISON TABLE")
print("="*60)
print(comparison_df.to_string(index=False))

TASK 3: CATEGORICAL NAIVE BAYES EVALUATION
Overall Model Accuracy: 90.00%

Confusion Matrix:
[[2 1]
 [0 7]]

Classification Report:
              precision    recall  f1-score   support

          No       1.00      0.67      0.80         3
         Yes       0.88      1.00      0.93         7

    accuracy                           0.90        10
   macro avg       0.94      0.83      0.87        10
weighted avg       0.91      0.90      0.89        10

TASK 4: SINGLE-SAMPLE INFERENCE (CATEGORICAL NAIVE BAYES)
Input Query: Outlook=Sunny, Temperature=Cool, Humidity=High, Wind=Strong
Predicted Class: No
Class Probabilities -> No: 0.7894, Yes: 0.2106

TASK 5: MODEL COMPARISON TABLE
                  Model Test Accuracy Sample Prediction  P(No) P(Yes)
Categorical Naive Bayes         90.0%                No 0.7894 0.2106
          Decision Tree        100.0%                No 1.0000 0.0000
    Logistic Regression         90.0%                No 0.8202 0.1798
 Support Vector Machine        

## Analysis Report
The model comparison demonstrates how fundamental algorithmic differences lead to varying probability confidence scores, even when all classifiers arrive at the same final class label (No):

Categorical Naive Bayes assumes strong conditional independence between features given the class label. By calculating feature-conditioned likelihoods independently and applying Bayes' Theorem (with Laplace smoothing), it yields a confident 79.5% probability for 'No', driven heavily by the combined negative evidence of Outlook=Sunny and Humidity=High.

Decision Tree creates deterministic partition boundaries based on information gain. The query sample maps directly into a pure leaf node trained on negative instances, resulting in an absolute, uncalibrated probability of 1.00 for 'No' (100%).

Logistic Regression and SVM use continuous linear decision boundaries in one-hot encoded vector space. Because the query instance lies relatively close to the decision boundary in a small 14-sample dataset, Logistic Regression assigns a milder 61.2% probability, while SVM (using Platt scaling for posterior probability estimation) assigns a conservative 58.3% probability.